# Technical Appendix: Beyond Correlation
## Case Study: Does Smoking Cause Lung Cancer?

In this notebook, we analyze the historical data from the **1954 British Doctors Study** by Richard Doll and Austin Bradford Hill. We explore how researchers used statistical logic to prove causality in a world without Randomized Controlled Trials (RCTs), ultimately debunking Sir Ronald Fisher's 'Hidden Gene' theory.

### Objectives:
1. **Dose-Response Analysis**: Testing if higher exposure leads to higher risk.
2. **Cornfield's Inequality**: A sensitivity analysis to test the plausibility of hidden confounders.
3. **Intervention Analysis**: Examining the effect of quitting smoking as a 'natural experiment'.

### Part 1: The Dose-Response Gradient
**Concept**: If the relationship is causal, we expect a 'gradient'. A binary hidden gene (present/absent) struggles to explain a smooth, steep increase in risk as smoking intensity increases. A biological toxin, however, follows this pattern perfectly.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Historical data (1954): Lung cancer death rates per 1,000 men per year
data_dose = {
    'Category': ['Non-Smokers', '1-14 cigs/day', '15-24 cigs/day', '25+ cigs/day'],
    'Death_Rate': [0.07, 0.47, 0.86, 1.66]
}

df_dose = pd.DataFrame(data_dose)

# Calculating Relative Risk (RR) relative to non-smokers
baseline = df_dose.iloc[0]['Death_Rate']
df_dose['Relative_Risk'] = df_dose['Death_Rate'] / baseline

print("Historical Dose-Response Table:")
print(df_dose)

# Visualization
plt.figure(figsize=(10, 5))
sns.barplot(x='Category', y='Death_Rate', data=df_dose, palette='Reds_d')
plt.title('Lung Cancer Death Rates by Smoking Intensity (Doll & Hill, 1954)')
plt.ylabel('Death Rate per 1,000 men/year')
plt.show()

**Result Interpretation**: The data shows a staggering increase. Heavy smokers face **23.7 times** the risk of lung cancer compared to non-smokers. The 'step-ladder' shape of the graph is a strong indicator of a biological causal mechanism (Dose-Response) rather than a simple hidden genetic binary.

### Part 2: Cornfield's Inequality (Sensitivity Analysis)
**Concept**: Jerome Cornfield proved mathematically that to 'explain away' a Relative Risk (RR) as a mere correlation caused by a hidden confounder ($U$), that confounder must be at least $RR$ times more prevalent in the exposed group than in the unexposed group.

In [ ]:
def cornfield_test(rr_observed):
    """
    Calculates the required prevalence ratio for a hidden confounder.
    """
    print(f"Observed Relative Risk: {rr_observed:.2f}")
    print(f"To nullify this effect, a hidden gene (U) must be {rr_observed:.2f} times more common in smokers.")
    print("--------------------------------------------------------------------------------")
    
    # Example Scenario
    p_unexposed = 0.04 # If 4% of non-smokers have the 'bad' gene
    required_p_exposed = p_unexposed * rr_observed
    
    if required_p_exposed > 1.0:
        print(f"IMPOSSIBLE: The gene would need to be present in {required_p_exposed*100:.1f}% of smokers,")
        print(f"which exceeds 100%. A hidden gene cannot explain this effect alone.")
    else:
        print(f"UNLIKELY: The gene would need to be present in {required_p_exposed*100:.1f}% of smokers.")

rr_heavy = df_dose.iloc[-1]['Relative_Risk']
cornfield_test(rr_heavy)

**Result Interpretation**: Since the Relative Risk is so high (23.7), the 'Hidden Gene' would need to be distributed in the population with near-perfect separation between smokers and non-smokers. No known genetic trait follows such a radical distribution, making Fisher's argument mathematically implausible.

### Part 3: The Natural Experiment (Quitting Smoking)
**Concept**: In Causal Inference (Pearl's Ladder of Causation), the ultimate test is *intervention*. While we couldn't force people to smoke, we could observe what happened when they *stopped*. If the cause was genetic, quitting wouldn't change the risk (genes don't change).

In [ ]:
# Quitting data: Risk reduction over time
data_quit = {
    'Status': ['Still Smoking', 'Quit < 5 yrs', 'Quit 5-9 yrs', 'Quit 10-14 yrs', 'Quit 15+ yrs', 'Never Smoked'],
    'Death_Rate': [1.25, 0.67, 0.49, 0.25, 0.18, 0.07]
}

df_quit = pd.DataFrame(data_quit)

plt.figure(figsize=(10, 5))
plt.plot(df_quit['Status'], df_quit['Death_Rate'], marker='o', linestyle='-', color='blue', linewidth=2)
plt.title('The Natural Intervention: Death Rates After Quitting')
plt.ylabel('Death Rate per 1,000 men/year')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

reduction = (1 - (df_quit.iloc[4]['Death_Rate'] / df_quit.iloc[0]['Death_Rate'])) * 100
print(f"Risk reduction after 15+ years of quitting: {reduction:.1f}%")

**Result Interpretation**: The dramatic 85%+ reduction in risk among former smokers is the 'smoking gun'. Because the intervention (quitting) changed the outcome (death rate), we can move beyond correlation into the realm of **confirmed causality**.